In [1]:
import pandas as pd
import numpy as np 


df1=pd.read_csv("/home/saidharahas/jupyter_projects/anime_reco/anime_data/anime.csv")
df2=pd.read_csv("/home/saidharahas/jupyter_projects/anime_reco/anime_data/rating.csv")
df1.head()


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [3]:
df2.head()

,user_id,anime_id,rating
0,1,20,-1
1,1,24,-1
2,1,79,-1
3,1,226,-1
4,1,241,-1


In [4]:
print(df1.shape)
print(df2.shape)

(12294, 7)
(7813737, 3)


In [35]:
from collections import defaultdict

genre=set()
gen_counts=defaultdict(int)
for i in df1["genre"]:
      j=str(i).split(',')
      for k in j:
            genre.add(k.strip())
            gen_counts[k.strip()]+=1
genre=list(genre)
genre.sort()
for i in genre:
       print(i)
        

Action
Adventure
Cars
Comedy
Dementia
Demons
Drama
Ecchi
Fantasy
Game
Harem
Hentai
Historical
Horror
Josei
Kids
Magic
Martial Arts
Mecha
Military
Music
Mystery
Parody
Police
Psychological
Romance
Samurai
School
Sci-Fi
Seinen
Shoujo
Shoujo Ai
Shounen
Shounen Ai
Slice of Life
Space
Sports
Super Power
Supernatural
Thriller
Vampire
Yaoi
Yuri
nan


In [30]:
print(gen_counts)

defaultdict(<class 'int'>, {'Drama': 2016, 'Romance': 1464, 'School': 1220, 'Supernatural': 1037, 'Action': 2845, 'Adventure': 2348, 'Fantasy': 2309, 'Magic': 778, 'Military': 426, 'Shounen': 1712, 'Comedy': 4645, 'Historical': 806, 'Parody': 408, 'Samurai': 148, 'Sci-Fi': 2070, 'Thriller': 87, 'Sports': 543, 'Super Power': 465, 'Space': 381, 'Slice of Life': 1220, 'Mecha': 944, 'Music': 860, 'Mystery': 495, 'Seinen': 547, 'Martial Arts': 265, 'Vampire': 102, 'Shoujo': 603, 'Horror': 369, 'Police': 197, 'Psychological': 229, 'Demons': 294, 'Ecchi': 637, 'Josei': 54, 'Shounen Ai': 65, 'Game': 181, 'Dementia': 240, 'Harem': 317, 'Cars': 72, 'Kids': 1609, 'Shoujo Ai': 55, 'nan': 62, 'Hentai': 1141, 'Yaoi': 39, 'Yuri': 42})


In [20]:
df1.shape

(12294, 7)

In [21]:
df2.shape

(7813737, 3)

In [36]:
for i in genre:
     val=list()
     for j in df1['genre']:
          words=str(j).split(',')
          for k in words:
               k.strip()
          if i in words:
              val.append(1)
          else:
              val.append(0)
     df1[i]=val               
          
  

In [37]:
df1.head()

,anime_id,name,genre,type,episodes,rating,members,Action,Adventure,Cars,...,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Vampire,Yaoi,Yuri,nan
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
from scipy.sparse import csr_matrix

users = df2["user_id"].astype("category").cat.codes
anime = df2["anime_id"].astype("category").cat.codes

mat = csr_matrix(
    (df2["rating"], (users, anime))
)
       
    


In [16]:
from sklearn.decomposition import NMF 
nmf = NMF(
    n_components=50,
    init="random",
    random_state=42,
    max_iter=500
)
W = nmf.fit_transform(mat)
H = nmf.components_

print(W.shape)
print(H.shape)
i=41
j=20

pred = W[i] @ H[:, j]

print(pred)

(69600, 50)
(50, 9927)
0.0


In [17]:
i=40
j=20

pred = W[i] @ H[:, j]

print(pred)

0.7442605111151966


In [18]:
i=41
j=28

pred = W[i] @ H[:, j]

print(pred)

0.010302349130384941


In [ ]:
def grad_step(i, j, lr):
    
      for epoch in range(100):
    
        for row in df2.itertuples():
    
            i = row.user_id
            j = row.anime_id
            r = row.rating
    
            pred = W[i] @ H[:, j]
    
            err = r - pred
    
            old_w = W[i].copy()
    
            W[i] += lr * err * H[:, j]
    
            H[:, j] += lr * err * old_w
            print("OLD W:")
            print(old_w)
        
            print("\nNEW W:")
            print(W[i])
        
            print("\nCHANGE IN W:")
            print(W[i] - old_w)
        
            print("\nOLD H:")
            print(old_h)
        
            print("\nNEW H:")
            print(H[:, j])
        
            print("\nCHANGE IN H:")
            print(H[:, j] - old_h)
        
            print("\nPrediction:", pred)
            print("Error:", err)


In [ ]:
grad_step(41,28,0.0005)

In [30]:
##batch incrementation of ratings training 
t_users=df2["user_id"].nunique()
print(t_users)
t_anime=df1["anime_id"].nunique()


t_anime= df2["anime_id"].max()

print(t_anime)
k=10
W = np.random.normal(0, 0.01, (t_users+1, k))
H = np.random.normal(0, 0.01, (k, t_anime+1))
lr = 0.05
reg = 0.02

def update(st: int, bs: int):

    ldf = df2.iloc[st:st + bs]

    for epochs in range(50):

        total_err = 0

        for row in ldf.itertuples():

            j = row.anime_id
            i = row.user_id
            r = row.rating

            if j > 73515:
                continue

            if i > 12294:
                continue

            pred = W[i] @ H[:, j]
            err = r - pred

            total_err += err * err

            old_w = W[i].copy()

            W[i] += lr * (err * H[:, j] - reg * W[i])
            H[:, j] += lr * (err * old_w - reg * H[:, j])

        print(f"Epoch {epochs + 1}")
        print("Loss:", total_err)
        print("-" * 30)
            
         
                                 
update(0,1000)



73515
34519
Epoch 1
Loss: 29095.450719027504
------------------------------
Epoch 2
Loss: 26446.695268579413
------------------------------
Epoch 3
Loss: 12173.158926758751
------------------------------
Epoch 4
Loss: 3757.560894509856
------------------------------
Epoch 5
Loss: 1737.4853962404381
------------------------------
Epoch 6
Loss: 848.4609069821398
------------------------------
Epoch 7
Loss: 430.94342443730227
------------------------------
Epoch 8
Loss: 179.40053559733357
------------------------------
Epoch 9
Loss: 112.5797192804274
------------------------------
Epoch 10
Loss: 86.11232405506419
------------------------------
Epoch 11
Loss: 69.33597354878593
------------------------------
Epoch 12
Loss: 57.804713485415384
------------------------------
Epoch 13
Loss: 50.09090057076619
------------------------------
Epoch 14
Loss: 44.97536142175025
------------------------------
Epoch 15
Loss: 41.36857336464563
------------------------------
Epoch 16
Loss: 38.560252185857

In [27]:
i=df2.iloc[1001].user_id
j=df2.iloc[1001].anime_id
r=df2.iloc[1001].rating
pred = W[i] @ H[:, j]
print("actual rating"+str(r))
print("prediction"+str(pred))

actual rating8
prediction7.5716648017836805


In [6]:
print(df2["rating"].mean())

6.144029546937656


In [44]:
import numpy as np

# choose reference anime (example: index 0)
target = H[:, 5114]

best_sim = -1
best_ind = -1

for i in range(H.shape[1]):  # 3400 anime
    if i==5114:
         continue
    vec = H[:, i]

    sim = np.dot(vec, target) / (
        np.linalg.norm(vec) * np.linalg.norm(target)
    )

    if sim > best_sim:
        best_sim = sim
        best_ind = i




print("Most similar anime:", df1["anime_id"].iloc[best_ind])
print(df1[df1["anime_id"]==df1["anime_id"].iloc[best_ind]])
print(df1[df1["anime_id"]==5114])
          
          


Most similar anime: 12611
      anime_id                name                     genre type episodes  \
5153     12611  Sengoku Collection  Fantasy, Parody, Samurai   TV       26   

      rating  members  
5153    6.54    16287  
   anime_id                              name  \
1      5114  Fullmetal Alchemist: Brotherhood   

                                               genre type episodes  rating  \
1  Action, Adventure, Drama, Fantasy, Magic, Mili...   TV       64    9.26   

   members  
1   793665  


In [45]:
df1[df1["anime_id"]==df1["anime_id"].iloc[best_ind]]

,anime_id,name,genre,type,episodes,rating,members
5153,12611,Sengoku Collection,"Fantasy, Parody, Samurai",TV,26,6.54,16287


In [51]:
df2[df2["rating"]].shape

NameError: name 'NaN' is not defined

In [2]:
anime_train = set(df2["anime_id"].unique())
anime_meta  = set(df1["anime_id"].unique())

print("In ratings but not anime.csv:",
      len(anime_train - anime_meta))

print("In anime.csv but not ratings:",
      len(anime_meta - anime_train))

In ratings but not anime.csv: 3
In anime.csv but not ratings: 1097


In [3]:
print(df1[df1["name"].str.contains("Tokyo Ghoul", case=False, na=False)])

      anime_id                            name  \
449      22319                     Tokyo Ghoul   
1048     30458   Tokyo Ghoul: &quot;Jack&quot;   
1268     31297  Tokyo Ghoul: &quot;Pinto&quot;   
1518     27899                  Tokyo Ghoul √A   

                                                  genre type episodes  rating  \
449   Action, Drama, Horror, Mystery, Psychological,...   TV       12    8.07   
1048  Action, Drama, Horror, School, Seinen, Superna...  OVA        1    7.71   
1268  Action, Drama, Horror, Mystery, Psychological,...  OVA        1    7.61   
1518  Action, Drama, Horror, Mystery, Psychological,...   TV       12    7.52   

      members  
449    618056  
1048    70635  
1268    52312  
1518   408357  
